# Laboratorio de IA ética: modelo base del Titanic

Predecir la supervivencia de pasajeros con un bosque aleatorio. Ejecuta las celdas en orden.
El preprocesamiento aprende únicamente del conjunto de entrenamiento; la evaluación utiliza validación.
El conjunto de prueba queda reservado para una evaluación final posterior.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/memo124/Laboratorio_IA_etica/blob/main/main.ipynb)

## 1. Dependencias

Este notebook utiliza las librerías incluidas en Google Colab.
Todos los imports se concentran en la siguiente celda.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 2. Configuración

Rutas, columnas y parámetros compartidos del experimento.

In [ ]:
DATA_PATH = Path("train.csv")
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
RANDOM_STATE = 42
TARGET_COLUMN = "Survived"
NUMERIC_COLUMNS = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL_COLUMNS = ["Pclass", "Sex", "Embarked"]
TEST_FRACTION = 0.20
VALIDATION_FRACTION = 0.20
N_ESTIMATORS = 100

## 3. Carga de datos

Se utiliza el CSV local. Si no existe, se descarga y se guarda para próximas ejecuciones.

In [ ]:
def load_dataset(path: Path, url: str) -> pd.DataFrame:
    """Lee los datos locales o descarga y guarda una copia del CSV."""
    if path.exists():
        return pd.read_csv(path)

    dataset = pd.read_csv(url)
    dataset.to_csv(path, index=False)
    return dataset


data = load_dataset(DATA_PATH, DATA_URL)

## 4. Exploración inicial

Vista previa, tipos de datos y valores faltantes por columna.

In [ ]:
display(data.head())
data.info()
display(data.isna().sum().rename("Valores faltantes").to_frame())

## 5. Separación de entradas y etiqueta

`Survived` es la etiqueta: 1 indica supervivencia y 0 indica fallecimiento. Se conservan todas las demás columnas, incluyendo `Ticket`, para compartir las mismas particiones entre integrantes. El preprocesador seleccionará únicamente las siete variables del Baseline.

In [ ]:
features = data.drop(columns=[TARGET_COLUMN])
target = data[TARGET_COLUMN]

## 6. División de datos

Entrenamiento (60 %), validación (20 %) y prueba (20 %). La estratificación conserva aproximadamente la proporción de supervivientes en cada conjunto.

In [ ]:
X_remaining, X_test, y_remaining, y_test = train_test_split(
    features,
    target,
    test_size=TEST_FRACTION,
    stratify=target,
    random_state=RANDOM_STATE,
)

# La validación representa el 25 % del 80 % restante: un 20 % del total.
X_train, X_val, y_train, y_val = train_test_split(
    X_remaining,
    y_remaining,
    test_size=VALIDATION_FRACTION / (1 - TEST_FRACTION),
    stratify=y_remaining,
    random_state=RANDOM_STATE,
)

split_sizes = pd.Series(
    {"Entrenamiento": len(X_train), "Validación": len(X_val), "Prueba": len(X_test)},
    name="Filas",
)
split_summary = split_sizes.to_frame()
split_summary["Porcentaje"] = (100 * split_sizes / len(data)).round(1)
display(split_summary)

# Verificamos que las particiones compartidas conserven filas y etiquetas alineadas.
partitions = [(X_train, y_train), (X_val, y_val), (X_test, y_test)]
for partition_features, partition_target in partitions:
    assert partition_features.index.equals(partition_target.index)
    assert "Ticket" in partition_features.columns
    assert TARGET_COLUMN not in partition_features.columns

partition_indices = [set(partition_features.index) for partition_features, _ in partitions]
assert partition_indices[0].isdisjoint(partition_indices[1])
assert partition_indices[0].isdisjoint(partition_indices[2])
assert partition_indices[1].isdisjoint(partition_indices[2])
assert set.union(*partition_indices) == set(data.index)
assert sum(len(partition_features) for partition_features, _ in partitions) == len(data)


## 7. Preprocesamiento

Las variables numéricas se imputan con la mediana; las categóricas, con la moda, y luego se codifican con one-hot. `Pclass` se trata como categoría. Esta celda solo define las transformaciones.

In [ ]:
numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, NUMERIC_COLUMNS),
        ("categorical", categorical_transformer, CATEGORICAL_COLUMNS),
    ],
    verbose_feature_names_out=False,
)

## 8. Construcción del modelo

Un único pipeline aplica el preprocesamiento y el clasificador, evitando repetir transformaciones manualmente al predecir.

In [ ]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=N_ESTIMATORS,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

## 9. Entrenamiento

El ajuste del preprocesamiento y del clasificador utiliza únicamente los datos de entrenamiento.

In [ ]:
baseline_model.fit(X_train, y_train)

feature_names = baseline_model.named_steps["preprocessor"].get_feature_names_out()
print(f"Variables resultantes: {len(feature_names)}")
print(", ".join(feature_names))

## 10. Preparación de validación y prueba

Aplicamos las reglas aprendidas durante el entrenamiento con `transform`, sin volver a ajustarlas. Las matrices preparadas conservan el orden de las filas de sus respectivas particiones. Transformar prueba no implica evaluarla: sus etiquetas no se utilizan aquí.


In [ ]:
fitted_preprocessor = baseline_model.named_steps["preprocessor"]
X_val_prepared = fitted_preprocessor.transform(X_val)
X_test_prepared = fitted_preprocessor.transform(X_test)

# Ambas matrices deben tener las mismas variables que recibió el clasificador.
expected_features = baseline_model.named_steps["classifier"].n_features_in_
for prepared, original in [(X_val_prepared, X_val), (X_test_prepared, X_test)]:
    assert prepared.shape == (len(original), expected_features)
    assert not pd.isna(prepared).any()

print(f"Validación preparada: {X_val_prepared.shape}")
print(f"Prueba preparada: {X_test_prepared.shape}")


## 11. Evaluación en validación

El F1 y el reporte de clasificación establecen la referencia del modelo base. El conjunto de prueba ya está preparado, pero no se utiliza para evaluar el modelo ni tomar decisiones en esta etapa.

In [ ]:
y_val_pred = baseline_model.predict(X_val)
validation_f1 = f1_score(y_val, y_val_pred)

print(f"F1 en validación: {validation_f1:.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_val, y_val_pred))